# GPT-OSS-20B: best of 25 across four prior failure groups

**Hypothesis.** Sampling 25 responses with a 32,768-token output budget can recover
joint successes on problems that GPT-OSS-20B failed in the previous run. A joint
success requires both passing every supplied functional test and recovering the
exact secret message. “Best of 25” means oracle success: at least one of the 25
responses passes the chosen criterion; there is no learned ranking or selection.

**Selection.** Use four disjoint groups from the previous 100-problem run:

1. **Token limit:** `finish_reason == "length"`; select up to 20 problems. This
   includes empty and nonempty truncated outputs, regardless of prior grading.
2. **Encoding only:** functional tests passed, secret recovery failed; up to 25.
3. **Tests only:** functional tests failed, secret recovery passed; up to 25.
4. **Both:** functional tests and secret recovery both failed; up to 25.

The final three groups require a completed Modal verdict (`passed`, `failed`, or
`timeout`), no top-level grading error, and no prior token-limit finish. Thus
ungraded format/API errors are not labeled as two observed failures. For the
saved source run the available counts are **15, 9, 19, and 5**, respectively:
**48 problems × 25 responses = 1,200 requests**. The token-limit group includes all
12 Darkbloom cases previously identified as severe repetition. Include every
problem when a group is smaller than its cap; otherwise select uniformly without
replacement using the selection seed. Keep prompts, targets, cipher, and tests
unchanged. A problem occurs in exactly one group.

**Expected and alternative outcomes.** Positive joint pass@25 supports recovery
under repeated sampling at these settings. Zero joint pass@25 means none of the
25 attempts succeeded; it does not prove impossibility. Functional success
without joint success identifies remaining encoding failures. Empty, truncated,
or repetitive outputs may persist despite the larger token budget. Raw responses
retain provider, finish reason, token usage, and reasoning for inspection.

**Conclusions and limits.** These estimates describe selected prior failure groups,
not overall APPS performance. Changing both sample count and token budget does
not isolate either effect. Routing and sampling use provider defaults, so requests
are separate but independence is not guaranteed. As in the source runner, all new
attempts—including API and grading infrastructure errors—remain in the denominator
and errors are reported separately. These rates therefore include pipeline
reliability as well as model correctness. Group pass@k is the mean of per-problem
estimates, never an estimator computed from pooled candidate counts.

**Prerequisites and running.** Use the `stego` Conda environment as the Jupyter
kernel. Set `OPENROUTER_API_KEY` and `STEGO_ARTIFACTS_DIR`, configure Modal
credentials, and make the source run below available. Open this notebook anywhere
inside the repository and run cells in order. The generation cell submits paid
OpenRouter requests; code execution happens only in Modal. Generation uses batch
size 100, tqdm, no retries, and a 1,800-second request timeout. Batch progress may
stay at zero until the first batch returns.

**Outputs.** Each run creates a timestamped directory under
`$STEGO_ARTIFACTS_DIR/variable_naming_in_python_v2/experiment/E20260916_gpt_oss_20b_pass_at_k/`
with settings, selected queries, request IDs, raw generations, grades, and
per-problem and group summaries. Generation and grading JSONL records are flushed
as they become available to the notebook; responses still inside an API batch
cannot be checkpointed. Existing stage files are never overwritten. Rerun the
output-directory cell to start a new run; to regrade saved generations in a new
kernel, set `run_directory` to that existing artifact directory and skip generation
(the grading output must not already exist). Incomplete runs cannot produce a
complete pass@k report.

**Metric.** Reuse the repository's implementation of [Chen et al., Figure 3](https://arxiv.org/pdf/2107.03374).
For each problem and criterion, `n=25`; pass@1 is `c/25`, and pass@25 is `1` if at
least one candidate succeeds, otherwise `0`. No notebook tests are added.


In [ ]:
import json
import os
import random
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal, Self

import pandas as pd
from IPython.display import display
from pydantic import BaseModel, ConfigDict, Field, model_validator
from tqdm.auto import tqdm

repo_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "ciphers/variable_naming_in_python_v2/tinker/run_openrouter.py").is_file())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from ciphers.variable_naming_in_python_v2.data.codex_apps import pass_at_k
from ciphers.variable_naming_in_python_v2.decoder import CipherConfig
from ciphers.variable_naming_in_python_v2.tinker.run_openrouter import Query, check_openrouter_key, grade
from lib.utils.api_generator import APIGenerator

In [ ]:
class ExperimentConfig(BaseModel):
    """Persist sampling and execution settings for one repeated-sampling run.

    source_run is relative to STEGO_ARTIFACTS_DIR; selection_seed controls only
    problem selection, not generation. Model, sample counts, and k values fix
    this experiment's interface. Other fields control API batch size, output
    token limit, request timeout, and concurrent Modal evaluations.
    """

    model_config = ConfigDict(extra="forbid", frozen=True)
    source_run: Path = Path("tinker/20260916T181636150860Z")
    model: Literal["openai/gpt-oss-20b"] = "openai/gpt-oss-20b"
    selection_seed: int = 42
    token_limit_problem_cap: int = Field(default=20, ge=1)
    other_group_problem_cap: int = Field(default=25, ge=1)
    samples_per_problem: Literal[25] = 25
    ks: tuple[Literal[1], Literal[25]] = (1, 25)
    api_batch_size: int = Field(default=100, ge=1)
    max_tokens: int = Field(default=32_768, ge=1)
    timeout_s: int = Field(default=1_800, ge=1)
    modal_workers: int = Field(default=16, ge=1)

    @model_validator(mode="after")
    def require_artifact_relative_source(self) -> Self:
        """Reject source paths that escape the configured artifact root."""
        if self.source_run.is_absolute() or ".." in self.source_run.parts:
            raise ValueError("source_run must be relative to STEGO_ARTIFACTS_DIR")
        return self


config = ExperimentConfig()
artifact_root = (repo_root / os.environ["STEGO_ARTIFACTS_DIR"]).resolve()
source_directory = artifact_root / config.source_run

## Select prior failure groups and expand each problem to 25 requests

`queries.jsonl` is validated with the existing `Query` schema: `problem_id` joins
records, `prompt` goes to the model, `message_bits` is the target secret, and
`test_cases` contains private Modal inputs/outputs. Source `config.json` must
contain `cipher`, consumed by `CipherConfig` for decoding.

Source grading records must contain `model`, `problem_id`, `functional`, `secret`,
`joint`, `error`, and `modal`. The three booleans describe prior successes; `error`
marks generation/format/infrastructure errors; `modal` is null or a verdict whose
`status` distinguishes completed grading from runner failure. Source generation
records must contain `problem_id`, `model`, and `raw`, where `raw` is null or a
response with `choices[0].finish_reason`; only `length` selects the token-limit
group. `raw.provider` is optional diagnostic metadata. Sorting eligible IDs before
seeded sampling makes selection independent of source JSONL completion order.


In [ ]:
queries = [Query.model_validate_json(line) for line in (source_directory / "queries.jsonl").read_text().splitlines() if line.strip()]
query_by_id = {query.problem_id: query for query in queries}
if len(query_by_id) != len(queries):
    raise ValueError("Source queries contain duplicate problem IDs")
source_settings = json.loads((source_directory / "config.json").read_text())
cipher = CipherConfig.model_validate(source_settings["cipher"])
prior_results = [row for line in (source_directory / "openrouter-results.jsonl").read_text().splitlines() if line.strip() and (row := json.loads(line))["model"] == config.model]
prior_by_id = {row["problem_id"]: row for row in prior_results}
prior_generations = [json.loads(line) for line in (source_directory / "openrouter-gpt-oss-20b.jsonl").read_text().splitlines() if line.strip()]
prior_generation_by_id = {row["problem_id"]: row for row in prior_generations}
if len(prior_by_id) != len(prior_results) or set(prior_by_id) != set(query_by_id):
    raise ValueError("Expected exactly one prior 20B result per source problem")
if len(prior_generation_by_id) != len(prior_generations) or set(prior_generation_by_id) != set(query_by_id) or any(row["model"] != config.model for row in prior_generations):
    raise ValueError("Expected exactly one prior 20B generation per source problem")
if any(type(row[key]) is not bool for row in prior_results for key in ("functional", "secret", "joint")):
    raise ValueError("Prior success outcomes must be booleans")

eligible_by_group = {name: [] for name in ("token_limit", "encoding_only", "tests_only", "both")}
for problem_id in sorted(query_by_id):
    row = prior_by_id[problem_id]
    raw = prior_generation_by_id[problem_id]["raw"]
    finish_reason = raw["choices"][0]["finish_reason"] if raw is not None else None
    if finish_reason == "length":
        eligible_by_group["token_limit"].append(problem_id)
    elif row["error"] or row["modal"] is None or row["modal"]["status"] not in {"passed", "failed", "timeout"}:
        continue
    elif row["functional"] and not row["secret"]:
        eligible_by_group["encoding_only"].append(problem_id)
    elif not row["functional"] and row["secret"]:
        eligible_by_group["tests_only"].append(problem_id)
    elif not row["functional"] and not row["secret"]:
        eligible_by_group["both"].append(problem_id)

rng = random.Random(config.selection_seed)
selected_by_group = {}
for group, eligible_ids in eligible_by_group.items():
    cap = config.token_limit_problem_cap if group == "token_limit" else config.other_group_problem_cap
    selected_by_group[group] = sorted(rng.sample(eligible_ids, min(cap, len(eligible_ids))))
group_by_id = {problem_id: group for group, ids in selected_by_group.items() for problem_id in ids}
selected_ids = [problem_id for ids in selected_by_group.values() for problem_id in ids]
if not selected_ids or len(selected_ids) != len(set(selected_ids)):
    raise ValueError("Selection must contain at least one problem, with no overlapping groups")
selected_queries = [query_by_id[problem_id] for problem_id in selected_ids]

# problem_id + sample_id disambiguates attempts; group is retained in every artifact.
requests = [
    {"group": group_by_id[query.problem_id], "problem_id": query.problem_id, "sample_id": sample_id, "prompt": query.prompt}
    for query in selected_queries
    for sample_id in range(config.samples_per_problem)
]
expected_keys = {(row["problem_id"], row["sample_id"]) for row in requests}
display(
    pd.DataFrame(
        [
            {"group": group, "available": len(eligible_by_group[group]), "selected": len(ids), "requests": len(ids) * config.samples_per_problem}
            for group, ids in selected_by_group.items()
        ]
    )
)
print(f"Prepared {len(requests)} requests for {len(selected_ids)} distinct problems")
display(
    pd.DataFrame([{"group": group_by_id[problem_id], **prior_by_id[problem_id]} for problem_id in selected_ids])[["group", "problem_id", "functional", "secret", "joint", "error"]]
)

In [ ]:
experiment_name = "E20260916_gpt_oss_20b_pass_at_k"
run_directory = artifact_root / "variable_naming_in_python_v2" / "experiment" / experiment_name / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_directory.mkdir(parents=True, exist_ok=False)
(run_directory / "config.json").write_text(
    json.dumps(
        {
            "experiment": config.model_dump(mode="json"),
            "cipher": cipher.model_dump(mode="json"),
            "eligible_by_group": eligible_by_group,
            "selected_by_group": selected_by_group,
            "prior_selected_results": [prior_by_id[problem_id] for problem_id in selected_ids],
        },
        indent=2,
    )
)
(run_directory / "queries.jsonl").write_text("".join(query.model_dump_json() + "\n" for query in selected_queries))
(run_directory / "requests.jsonl").write_text("".join(json.dumps(row) + "\n" for row in requests))
print("Saving to", run_directory)

## Generate and save raw responses

Each generation record contains `group` (the prior failure group), `problem_id` and zero-based `sample_id` (together
unique), `model`, `text` (final message content, or an empty string), `raw` (the full
LiteLLM response, or null on API failure), and `error` (API failure text, otherwise
null). `grade` consumes `problem_id`, `model`, `text`, and `error`; `raw` is retained
for inspection. Reasoning is typically at
`raw.choices[0].message.reasoning_content`. Tool-call arguments are preserved in
`raw` but are not substituted for final content, matching the original runner.
Prompts, provider routing, JSON handling, and sampling defaults are unchanged.

In [ ]:
check_openrouter_key()
generation_path = run_directory / "generations.jsonl"
with generation_path.open("x") as output:
    responses = APIGenerator().api_generate_streaming(
        [request["prompt"] for request in requests],
        model=f"openrouter/{config.model}",
        batch_size=config.api_batch_size,
        max_new_tokens=config.max_tokens,
        enable_tqdm=True,
        return_raw=True,
        num_retries=0,
        batch_completion_kwargs={"timeout": config.timeout_s},
    )
    for request, response in tqdm(
        zip(requests, responses, strict=True),
        total=len(requests),
        desc="Save 20B responses",
        unit="response",
    ):
        record = {
            "group": request["group"],
            "problem_id": request["problem_id"],
            "sample_id": request["sample_id"],
            "model": config.model,
            "text": "",
            "raw": None,
            "error": None,
        }
        if response is None or isinstance(response, Exception):
            record["error"] = f"APIGenerator failed: {response}"
        else:
            record["raw"] = response.model_dump(mode="json")
            record["text"] = response.choices[0].message.content or ""
        output.write(json.dumps(record) + "\n")
        output.flush()
print("Saved", generation_path)

## Grade saved responses with the existing Modal runner

Generation is already on disk before grading begins. The existing `grade` function
checks JSON formatting, decodes the secret, and sends nonblank code to Modal.
Malformed final content is unsuccessful. Grading results contain `problem_id`,
`model`, `functional`, `secret`, `joint`, `modal` (full verdict/logs or null), `error`,
`decode_error`, and `seconds`; this notebook adds `sample_id` and `group` to preserve the join and failure group.
`modal.status == "runner_error"` also counts as an error in the summary.

In [ ]:
generations = [json.loads(line) for line in (run_directory / "generations.jsonl").read_text().splitlines()]
generation_keys = [(row["problem_id"], row["sample_id"]) for row in generations]
if len(generation_keys) != len(expected_keys) or set(generation_keys) != expected_keys:
    raise RuntimeError("Incomplete or duplicate generations; inspect the saved file")
result_path = run_directory / "results.jsonl"
with result_path.open("x") as output:
    with ThreadPoolExecutor(max_workers=config.modal_workers) as pool:
        futures = {pool.submit(grade, record, queries=query_by_id, cipher=cipher): record for record in generations}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Modal grading", unit="response"):
            record = future.result()
            record["sample_id"] = futures[future]["sample_id"]
            record["group"] = futures[future]["group"]
            output.write(json.dumps(record) + "\n")
            output.flush()
print("Saved", result_path)

## Per-problem and group pass@1 and pass@25

Report functional and joint success separately. No new candidates are discarded.
`errors` counts truthy `error` or a Modal `runner_error`; decoder failures affect
joint success and remain in `decode_error` for inspection. Every problem must have
all 25 unique sample IDs before summaries are produced.

The first table has one row per problem and success criterion. The second table
averages those estimates within each prior failure group; its `problems` column
makes unequal group sizes explicit. Empty source groups have no estimates.


In [ ]:
results = [json.loads(line) for line in (run_directory / "results.jsonl").read_text().splitlines()]
result_keys = [(row["problem_id"], row["sample_id"]) for row in results]
if len(result_keys) != len(expected_keys) or set(result_keys) != expected_keys:
    raise RuntimeError("Incomplete or duplicate grading; cannot report pass@k")
summary = []
for problem_id in selected_ids:
    rows = [row for row in results if row["problem_id"] == problem_id]
    errors = sum(bool(row["error"]) or (row["modal"] is not None and row["modal"]["status"] == "runner_error") for row in rows)
    for criterion in ("functional", "joint"):
        successes = sum(row[criterion] for row in rows)
        summary.append(
            {
                "group": group_by_id[problem_id],
                "problem_id": problem_id,
                "criterion": criterion,
                "n": len(rows),
                "c": successes,
                "errors": errors,
                **{f"pass@{k}": float(pass_at_k(len(rows), successes, k)) for k in config.ks},
            }
        )
summary_frame = pd.DataFrame(summary)
print(summary_frame.to_string(index=False))
(run_directory / "summary.json").write_text(json.dumps(summary, indent=2))
summary_frame.to_csv(run_directory / "summary.csv", index=False)
group_summary = (
    summary_frame.groupby(["group", "criterion"], sort=False)
    .agg(problems=("problem_id", "count"), attempts=("n", "sum"), successes=("c", "sum"), errors=("errors", "sum"), **{f"pass@{k}": (f"pass@{k}", "mean") for k in config.ks})
    .reset_index()
)
print("\nMean per-problem pass@k within each prior failure group:")
print(group_summary.to_string(index=False))
group_summary.to_json(run_directory / "group-summary.json", orient="records", indent=2)
group_summary.to_csv(run_directory / "group-summary.csv", index=False)
print("All artifacts:", run_directory)